In [6]:
import sys
sys.executable

'e:\\Documents\\Learn GenAI\\RAG-based-Enterprise-Knowledge-Assistant\\rag-based-enterprise-knowledge-assistant-venv\\Scripts\\python.exe'

## Installations

In [8]:
!pip list

Package                                  Version
---------------------------------------- -----------
aiohappyeyeballs                         2.7.1
aiohttp                                  3.14.3
aiosignal                                1.4.0
annotated-doc                            0.0.5
annotated-types                          0.8.0
anyio                                    4.15.1
asttokens                                3.0.2
attrs                                    26.1.0
bcrypt                                   5.0.0
build                                    1.6.1
certifi                                  2026.7.22
charset-normalizer                       3.5.1
chromadb                                 1.5.9
click                                    8.5.0
colorama                                 0.4.6
comm                                     0.2.3
debugpy                                  1.8.22
distro                                   1.9.0
durationpy                               0.1

## Imports

In [10]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_classic.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


## Configurations

In [11]:
load_dotenv(dotenv_path="./.env")
PDF_PATH = "./USA_Employee_Handbook-Freely_Available.pdf"
CHROMADB_PATH = "./chromadb"
COLLECTION_NAME = "company-policy"

## Document Loader


In [15]:
pdfloader = PyPDFLoader(PDF_PATH)
pages = pdfloader.load()


In [16]:
avg_page_len = sum([len(page.page_content) for page in pages])/len(pages)
avg_page_len

1744.8529411764705

In [17]:
pages[5]

Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2019-01-25T16:37:28+00:00', 'moddate': '2019-01-25T16:37:28+00:00', 'source': './USA_Employee_Handbook-Freely_Available.pdf', 'total_pages': 34, 'page': 5, 'page_label': '6'}, page_content='■ Referrers are still eligible for rewards even if a candidate is hired at a later time or \ngets hired for another position. \n \nWho can be referred? \n \nWe have two conditions for candidates who can qualify you for our rewards. They \nshould: \n■ Have not applied to our company for at least a year. \n■ Be hired as permanent full- or part-time employees (not as temporary employees \nor contractors.) \n \nOur company may use an online form or a platform where employees may refer \ncandidates. You can also reach out directly to our [HR/recruiters/Talent Acquisition \nManager] with referrals. \n \nGenerally, we encourage you to check our open positions and consider your social \nnetworks and external networks as po

## Recursive Loader

In [18]:
textsplitter = RecursiveCharacterTextSplitter(
    chunk_size = 2000, chunk_overlap=200
)
docs = textsplitter.split_documents(pages)
len(docs)

40

## Embedding model - OpenAI

In [19]:
openai_embeddings = OpenAIEmbeddings(api_key=os.getenv("OPENAI_API_KEY"))
dict(openai_embeddings)

{'client': <openai.resources.embeddings.Embeddings at 0x2c257485be0>,
 'async_client': <openai.resources.embeddings.AsyncEmbeddings at 0x2c257486900>,
 'model': 'text-embedding-ada-002',
 'dimensions': None,
 'deployment': 'text-embedding-ada-002',
 'openai_api_version': None,
 'openai_api_base': None,
 'openai_api_type': None,
 'openai_proxy': None,
 'embedding_ctx_length': 8191,
 'openai_api_key': SecretStr('**********'),
 'openai_organization': None,
 'allowed_special': None,
 'disallowed_special': None,
 'chunk_size': 1000,
 'max_retries': 2,
 'request_timeout': None,
 'headers': None,
 'tiktoken_enabled': True,
 'tiktoken_model_name': None,
 'show_progress_bar': False,
 'model_kwargs': {},
 'skip_empty': False,
 'default_headers': None,
 'default_query': None,
 'retry_min_seconds': 4,
 'retry_max_seconds': 20,
 'http_client': None,
 'http_async_client': None,
 'check_embedding_ctx_length': True}

## Vector Database - ChromaDB

In [20]:
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=openai_embeddings,
    collection_name = COLLECTION_NAME,
    persist_directory=CHROMADB_PATH
)

## Retriever

In [21]:
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k":4
    }
)

## Chat Model

In [22]:
openai_chat_llm  = ChatOpenAI(api_key=os.getenv("OPENAI_API_KEY"),
                              model="gpt-5-nano", 
                              temperature = 0)
dict(openai_chat_llm)

{'name': None,
 'cache': None,
 'verbose': False,
 'callbacks': None,
 'tags': None,
 'metadata': {'lc_versions': {'langchain-core': '1.6.5',
   'langchain': '1.4.2',
   'langchain-openai': '1.6.6'}},
 'custom_get_token_ids': None,
 'rate_limiter': None,
 'disable_streaming': False,
 'output_version': None,
 'profile': {'name': 'GPT-5 Nano',
  'release_date': '2025-08-07',
  'last_updated': '2025-08-07',
  'open_weights': False,
  'max_input_tokens': 272000,
  'max_output_tokens': 128000,
  'text_inputs': True,
  'image_inputs': True,
  'audio_inputs': False,
  'video_inputs': False,
  'text_outputs': True,
  'image_outputs': False,
  'audio_outputs': False,
  'video_outputs': False,
  'reasoning_output': True,
  'tool_calling': True,
  'structured_output': True,
  'attachment': True,
  'temperature': False,
  'image_url_inputs': True,
  'pdf_inputs': True,
  'pdf_tool_message': True,
  'image_tool_message': True,
  'tool_choice': True,
  'tool_call_streaming': True,
  'reasoning_effor

## Prompt

In [23]:
prompttemplate = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a company knowledge assistant.

Answer the user's question using only the
provided context.

If the answer cannot be found in the context,
say that you do not have enough information.

Do not invent company policies.

Context:
{context}
"""
    ),
    (
        "human",
        "{question}"
    ),
])


## Format the retrived documents into context

In [24]:
def format_context(retrived_docs):
    return "\n\n".join(
        doc.page_content
        for doc in retrived_docs
    )
 

## RAG CHAIN

In [25]:
rag_chain = (
    {"context": retriever | format_context,
     "question": RunnablePassthrough()}
     | prompttemplate
     | openai_chat_llm
     | StrOutputParser()

)

## Test

In [27]:
answer = rag_chain.invoke(input = "Introduce the company name and mission and tell me about the policy to deal with abuse and harrassment in workplace")

In [28]:
answer

'- Company name and mission\n  - The provided context does not include a specific company name or mission statement. The text uses placeholders like [Company name] and [state mission statement or values], so I don’t have enough information to introduce an exact name or mission.\n\n- Policy to deal with abuse and harassment in the workplace\n  - Harassment overview\n    - Harassment is a broad term and can include actions like sabotaging someone’s work, unwanted advances, derogatory comments about ethnicity or religion, spreading rumors, or ridiculing someone in front of others.\n    - Sexual harassment is illegal and will be seriously investigated; if found guilty, the employee will be terminated.\n  - Reporting and support\n    - If you’re being harassed, you can talk to:\n      - Offenders (directly, for minor harassment cases; not appropriate with customers or stakeholders),\n      - Your manager (especially if customers, stakeholders, or team members are involved),\n      - HR (fee